In [3]:
import os
from pathlib import Path

# Check current directory and .env locations
print(f"Current working directory: {os.getcwd()}")
print(f".env exists in current dir: {Path('.env').exists()}")
print(f".env exists in parent dir: {Path('../.env').exists()}")
print(f"Project root .env exists: {Path('../../.env').exists()}")

Current working directory: /content
.env exists in current dir: False
.env exists in parent dir: False
Project root .env exists: False


In [2]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv('../.env')
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login(token=hf_token)
else:
    print("HF_TOKEN not found in environment variables. Please set it in the .env file.")

HF_TOKEN not found in environment variables. Please set it in the .env file.


In [1]:
# Cell 1 - Load dataset
from datasets import load_dataset

ds = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k")
print(ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…):   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 112165
    })
})


In [22]:
# Cell 2 - Inspect structure
sample = ds['train'][0]
print("Keys:", sample.keys())
print()
print("INPUT:", sample['input'])
print()
print("OUTPUT:", sample['output'])

Keys: dict_keys(['instruction', 'input', 'output'])

INPUT: I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!

OUTPUT: Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common symptom is dizziness or giddiness, which is made worse with movements. Accompanying nausea and vomiting are common. The condition is due to problem in the e

In [23]:
# Cell 3 - Random samples
import random

indices = random.sample(range(len(ds['train'])), 10)

for i in indices:
    sample = ds['train'][i]
    print("INPUT:", sample['input'])
    print(f"INPUT LENGTH: {len(sample['input'])}\n")
    print(f"OUTPUT: {sample['output'][:200]}")
    print(f"OUTPUT LENGTH: {len(sample['output'])}\n")
    print("-" * 60)

INPUT: hello im ruth joy olave 27years old...last week i undergone a complete medical exam one of the requirements for working abroad i was surprise for the results,the findings for my x ray is i have a FIBROTIC SCARRING RIGHT APEX.what is it all about?is there a medication or treatment for that disease?
INPUT LENGTH: 298

OUTPUT: HelloFibrotic scarring in right apical region of lung may be due to past infection like tuberculosis etc. Fibrosis is a healed stage and generally this stage doesn't require any treatment. You may nee
OUTPUT LENGTH: 466

------------------------------------------------------------
INPUT: Hi I have been experiencing some weird symptoms maybe typical of pregnancy.Ive had my period as normal in August but September it came early only lasted 4days,at the end of it,it faded to a brownish colour.I have also had white discharge that had no smell.I get bad back pain,and am now feeling like I have butterflies in my stomach.my stomach has grown rapidly,it is hard at th

In [30]:
import re

def clean_output(text):
    # Strip filler openings
    filler_starts = [
        r'^Hello[\w\s,]*Welcome to Chat\s?Doctor[.\s]*',
        r'^Hi[\w\s,]*Welcome to Chat\s?Doctor[.\s]*',
        r'^Thanks for (using|writing to) Chat\s?Doctor[.\s]*',
        r'^Hello and welcome to Chat\s?Doctor[.\s]*',
        r'^and I hope I can help you today\.?[\s]*',
        r'^Thank you for (posting|consulting|writing|using)[\w\s,]*[.\s]+',
        r'^Hello[\s,]+',
        r'^Hi[\s,]+',
        r'^HI[\s,]+',
        r'^Dear[\w\s,]+,[\s]*',
    ]
    for pattern in filler_starts:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE).strip()

    # Strip trailing sign-offs
    filler_ends = [
        r'[,.]?\s*Best wishes,?\s*Chat\s?Doctor\.?$',
        r'[,.]?\s*I hope (this|it) helps?\.?$',
        r'[,.]?\s*I hope (this|it) (answers?|address(es)?) your (query|question)\.?$',
        r'[,.]?\s*Take care\.?$',
        r'[,.]?\s*Chat\s?Doctor\.?$',
        r'[,.]?\s*Best wishes\.?$',
        r'[,.]?\s*Hope this helps\.?$',
    ]
    for pattern in filler_ends:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE).strip()

    return text


def is_clean(sample):
    input_text  = sample['input']
    output_text = sample['output']

    if re.search(r'chatdoctor', input_text, re.IGNORECASE):
        return False
    if len(output_text.split()) < 30:
        return False
    if len(input_text.split()) + len(output_text.split()) > 600:
        return False
    return True


def is_clean_after_mapping(sample):
    if re.search(r'chat\s?doctor', sample['output'], re.IGNORECASE):
        return False
    # Reject if output still starts with common filler after cleaning
    if re.match(r'^(and |thank you|hello|hi\b)', sample['output'], re.IGNORECASE):
        return False
    return True


# Run full pipeline fresh
clean_data = ds['train'].filter(is_clean)
clean_data = clean_data.map(lambda s: {'output': clean_output(s['output'])})
clean_data = clean_data.filter(is_clean_after_mapping)

print(f"Final clean rows: {len(clean_data)}")

# Spot check 5 outputs
for i in range(5):
    print(f"OUTPUT: {clean_data[i]['output'][:300]}")
    print("-" * 60)

Filter:   0%|          | 0/112165 [00:00<?, ? examples/s]

Map:   0%|          | 0/107068 [00:00<?, ? examples/s]

Filter:   0%|          | 0/107068 [00:00<?, ? examples/s]

Final clean rows: 45205
OUTPUT: I would suggest that you see your doctor. Your baby maybe having bronchiolitis which is a lung infection common to your kids age. It is commonly caused by a virus. Albuterol via nebulization should be utilized in order to alleviate the wheezing and also help with the congestion. A decongestant can a
------------------------------------------------------------
OUTPUT: From history it seems that you might be having degenerative changes in your lower back spines giving pinched nerve pressure. There might be having osteomalacia or osteoporosis as well. Go for x-ray lumbosacral region for osteoarthritis. Physiotherapy like back extension exercises will be much helpfu
------------------------------------------------------------
OUTPUT: You have sustained twisting injury to foot which may lead to sprain or minor fracture in foot bones. Diagnosis needs to be confirmed by doing x-ray of the involved foot anteroposterior and oblique views to rule out bony abnormal

In [31]:
import random
random.seed(42)

indices = random.sample(range(len(clean_data)), 10000)
sampled = clean_data.select(indices)
print(f"Sampled: {len(sampled)} rows")

Sampled: 10000 rows


In [32]:
def format_sample(sample):
    system = "If you are a doctor, please answer the medical questions based on the patient's description."

    return {
        "text": (
            f"<|begin_of_text|>"
            f"<|start_header_id|>system<|end_header_id|>\n"
            f"{system}\n"
            f"<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n"
            f"{sample['input']}\n"
            f"<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n"
            f"{sample['output']}\n"
            f"<|eot_id|>"
        )
    }

formatted = sampled.map(format_sample)

# Verify
print(formatted[0]['text'])

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>
If you are a doctor, please answer the medical questions based on the patient's description.
<|eot_id|><|start_header_id|>user<|end_header_id|>
Hi! So I had four of wisdom teeth extracted over a year ago. Ever since then I ve had swellings and been on antibiotics on and off. But now it s worst because now I have a lump on my lower left jaw. The lump does not show up on a x-ray so my dental surgeon concluded that it has nothing to do with my teeth. I then did a ct scan recently. I haven t got back the actual scans but my pcp read the written results which she is unclear on, she said something to the effect that there is a density of liquid under the jaw. What does that mean? And in the mean time is the anything I can take to bring the swelling. Also it doesn t hurt. Just when I touch it there slight pain. I can t yawn too hard then it ll hurt. I can t chew gum or eat anything on the left said that s hard like crackers (mostly b

In [34]:
from huggingface_hub import login
from google.colab import userdata

# Retrieve your secret token
hf_token = userdata.get('collab-project-token')

if hf_token:
  login(token=hf_token)

In [35]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
tokenizer.add_special_tokens({'pad_token': '<pad>'})

lengths = [
    len(tokenizer(s['text'], truncation=False)['input_ids'])
    for s in formatted
]

print(f"Shortest:    {min(lengths)} tokens")
print(f"Longest:     {max(lengths)} tokens")
print(f"Average:     {sum(lengths) / len(lengths):.0f} tokens")
print(f"Over 512:    {sum(1 for l in lengths if l > 512)}")
print(f"Over 1024:   {sum(1 for l in lengths if l > 1024)}")
print(f"Over 2048:   {sum(1 for l in lengths if l > 2048)}")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Shortest:    78 tokens
Longest:     794 tokens
Average:     261 tokens
Over 512:    110
Over 1024:   0
Over 2048:   0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [36]:
split = formatted.train_test_split(test_size=0.1, seed=42)

train_dataset = split['train']
eval_dataset  = split['test']

print(f"Train: {len(train_dataset)}")
print(f"Eval:  {len(eval_dataset)}")

Train: 9000
Eval:  1000


In [37]:
split.save_to_disk("data/medquad_formatted")
print("Saved.")

Saving the dataset (0/1 shards):   0%|          | 0/9000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saved.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')